# Simulation for doctor agent 2.1 with partial fine-tuning

Files needed:
*   Base doctor agent that was saved locally
*   Generated patients (from llama3_8B, only original)


Unzip generated patients

In [ ]:
!unzip ./llama3_8B_original.zip

Archive:  ./llama3_8B_original.zip
  inflating: llama3_8B_original/heart_disease_patients/patients_wegeners_granulomatosis.json  
  inflating: llama3_8B_original/heart_disease_patients/patients_cardiomyopathy.json  
  inflating: llama3_8B_original/heart_disease_patients/patients_aortic_aneurysm.json  
  inflating: llama3_8B_original/heart_disease_patients/patients_chronic_hypotension.json  
  inflating: llama3_8B_original/heart_disease_patients/patients_thrombophlebitis.json  
  inflating: llama3_8B_original/heart_disease_patients/patients_acute_rheumatic_fever.json  
  inflating: llama3_8B_original/heart_disease_patients/patients_cardiac_hypertrophy.json  
  inflating: llama3_8B_original/heart_disease_patients/patients_pulmonary_heart_disease.json  
  inflating: llama3_8B_original/heart_disease_patients/patients_ischaemic_heart_disease.json  
  inflating: llama3_8B_original/heart_disease_patients/patients_cardiovascular_disease.json  
  inflating: llama3_8B_original/heart_disease_pati

Edit the following path to match the paths to the patients generated with llama3:

In [ ]:
folder_path_llama3_heart = './llama3_8B_original/heart_disease_patients'
folder_path_llama3_infection = './llama3_8B_original/infection_disease_patients'

Read all patients generated from mistral

In [ ]:
import os
import json

folder_path = folder_path_llama3_heart
json_contents = []

for filename in os.listdir(folder_path):
    if filename.endswith('.json'):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, 'r', encoding='utf-8') as f:
            content = json.load(f)
            json_contents.append(content)  # Append the parsed JSON to your list

print(len(json_contents))

folder_path = folder_path_llama3_infection
json_contents2 = []

for filename in os.listdir(folder_path):
    if filename.endswith('.json'):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, 'r', encoding='utf-8') as f:
            content = json.load(f)
            json_contents2.append(content)  # Append the parsed JSON to your list

print(len(json_contents2))

99
102


Data handling

In [ ]:
import pandas as pd

df = pd.DataFrame()

for i in range(len(json_contents)):
    json_file = json_contents[i]
    for j in range(len(json_file)):
        json_file_elem = json_file[j]
        patients_df = pd.DataFrame(json_file_elem["patients"])
        df = pd.concat([df, patients_df], ignore_index=True)

df2 = pd.DataFrame()

for i in range(len(json_contents2)):
    json_file = json_contents2[i]
    for j in range(len(json_file)):
        json_file_elem = json_file[j]
        patients_df = pd.DataFrame(json_file_elem["patients"])
        df2 = pd.concat([df2, patients_df], ignore_index=True)


total_df = pd.concat([df, df2], ignore_index=True)
total_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3404 entries, 0 to 3403
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Full name              3404 non-null   object
 1   Age                    3404 non-null   object
 2   Gender                 3404 non-null   object
 3   Symptoms               3404 non-null   object
 4   Symptom duration       3404 non-null   object
 5   Recent travel history  3404 non-null   object
 6   Medical history        3404 non-null   object
 7   Mixed diseases         3404 non-null   object
 8   Correct disease        3404 non-null   object
 9   Diagnosis cause        3404 non-null   object
 10  patient_number         3404 non-null   object
dtypes: object(11)
memory usage: 292.7+ KB


In [ ]:
import pandas as pd
import ast  # for safely evaluating strings that represent lists

# Define transformation functions
def build_question(row):
    return (
        f"This patient is a {row['Age']} year old {row['Gender']} presenting with the following symptoms: "
        f"{row['Symptoms']}. Symptoms have lasted for {row['Symptom duration']}. "
        f"Recent travel history: {row['Recent travel history']}. "
        f"Medical history: {row['Medical history']}. "
        "What is the disease of this patient?"
    )

def build_options(diseases):
    return {chr(65 + i): disease for i, disease in enumerate(diseases)}

def get_answer_idx(options, correct_disease):
    for key, val in options.items():
        if val == correct_disease:
            return key
    return None

# Apply transformations
total_df["question"] = total_df.apply(build_question, axis=1)
total_df["options"] = total_df["Mixed diseases"].apply(build_options)
total_df["answer"] = total_df["Correct disease"]
total_df["answer_idx"] = total_df.apply(lambda row: get_answer_idx(row["options"], row["answer"]), axis=1)

# Keep only the required columns
final_df = total_df[["question", "options", "answer", "answer_idx"]].copy()
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3404 entries, 0 to 3403
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   question    3404 non-null   object
 1   options     3404 non-null   object
 2   answer      3404 non-null   object
 3   answer_idx  3404 non-null   object
dtypes: object(4)
memory usage: 106.5+ KB


In [ ]:
final_df.head(10)
final_df["answer_idx"].value_counts(normalize=True)

,proportion
answer_idx,
C,0.217098
B,0.199177
E,0.197415
A,0.195946
D,0.190364


In [ ]:
! pip3 install transformers datasets torch accelerate evaluate

# Now let's get serious

Load model and tokenizer

In [ ]:
!unzip ./epoch3.zip  # Unzip zipped model

Archive:  ./epoch3.zip
   creating: lr_5e-05/checkpoint-94551/
  inflating: lr_5e-05/checkpoint-94551/optimizer.pt  
  inflating: lr_5e-05/checkpoint-94551/tokenizer_config.json  
  inflating: lr_5e-05/checkpoint-94551/model.safetensors  
  inflating: lr_5e-05/checkpoint-94551/vocab.txt  
  inflating: lr_5e-05/checkpoint-94551/training_args.bin  
  inflating: lr_5e-05/checkpoint-94551/rng_state.pth  
  inflating: lr_5e-05/checkpoint-94551/scheduler.pt  
  inflating: lr_5e-05/checkpoint-94551/special_tokens_map.json  
  inflating: lr_5e-05/checkpoint-94551/scaler.pt  
  inflating: lr_5e-05/checkpoint-94551/config.json  
  inflating: lr_5e-05/checkpoint-94551/trainer_state.json  
  inflating: lr_5e-05/checkpoint-94551/tokenizer.json  


Edit the model_path variable to match the path of your pre-trained model:

In [ ]:
from transformers import BertTokenizer, BertForMultipleChoice

model_path = "./lr_5e-05/checkpoint-94551/"

tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForMultipleChoice.from_pretrained(model_path)

Some weights of BertForMultipleChoice were not initialized from the model checkpoint at ./lr_5e-05/checkpoint-94551/ and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Partial fine-tuning : only update parameters from the last two transformer blocks and the classifying head

In [ ]:
# Step 1: Freeze everything
for param in model.parameters():
    param.requires_grad = False

# Step 2: Unfreeze last 2 transformer blocks
for layer in model.bert.encoder.layer[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

# Step 3: Unfreeze classifier head
for param in model.classifier.parameters():
    param.requires_grad = True

# Step 4: Function to print trainable/total params
def print_trainable_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Total parameters: {total_params:,}")
    print(f"Fraction trainable: {trainable_params / total_params:.6f}")

print_trainable_parameters(model)

bert.encoder.layer.10.attention.self.query.weight
bert.encoder.layer.10.attention.self.query.bias
bert.encoder.layer.10.attention.self.key.weight
bert.encoder.layer.10.attention.self.key.bias
bert.encoder.layer.10.attention.self.value.weight
bert.encoder.layer.10.attention.self.value.bias
bert.encoder.layer.10.attention.output.dense.weight
bert.encoder.layer.10.attention.output.dense.bias
bert.encoder.layer.10.attention.output.LayerNorm.weight
bert.encoder.layer.10.attention.output.LayerNorm.bias
bert.encoder.layer.10.intermediate.dense.weight
bert.encoder.layer.10.intermediate.dense.bias
bert.encoder.layer.10.output.dense.weight
bert.encoder.layer.10.output.dense.bias
bert.encoder.layer.10.output.LayerNorm.weight
bert.encoder.layer.10.output.LayerNorm.bias
bert.encoder.layer.11.attention.self.query.weight
bert.encoder.layer.11.attention.self.query.bias
bert.encoder.layer.11.attention.self.key.weight
bert.encoder.layer.11.attention.self.key.bias
bert.encoder.layer.11.attention.self.val

In [ ]:
from transformers import AutoTokenizer
from tqdm import tqdm

def preprocess_medqa(examples):
    """Tokenize question and choices separately for multiple-choice classification"""
    inputs = {"input_ids": [], "attention_mask": [], "token_type_ids": [], "labels": []}

    for example in tqdm(examples, total=len(examples)):
    #for i in range(len(examples["question"])):  # Process each example individually
        question = example["question"]
        options = example["options"] # Dictionary {'A': 'Ampicillin', 'B': 'Ceftriaxone', ...}
        correct_answer = example["answer_idx"]  # Single letter ('A', 'B', ...)

        # Ensure consistent option order (sort by key)
        option_keys = sorted(options.keys())
        option_values = [options[key] for key in option_keys]  # List of answer choices

        # Convert correct answer letter to index
        if correct_answer in option_keys:
            label = option_keys.index(correct_answer)  # Map the letter to index (0, 1, ...)
        else:
            raise ValueError(f"Unexpected answer key: {correct_answer} in {option_keys}")

        # Tokenize question-answer pairs
        encoding = tokenizer(
            [question] * len(option_values),  # Repeat the question for each choice
            option_values,  # List of choices
            padding="max_length",
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        # Append results to inputs dictionary
        inputs["input_ids"].append(encoding["input_ids"].squeeze(0))  # Remove extra batch dimension
        inputs["attention_mask"].append(encoding["attention_mask"].squeeze(0))
        inputs["token_type_ids"].append(encoding.get("token_type_ids", None).squeeze(0) if "token_type_ids" in encoding else None)
        inputs["labels"].append(label)  # Correct answer index

    return inputs

In [ ]:
import torch
from datasets import Dataset, DatasetDict

all_data = Dataset.from_pandas(final_df)
all_data_preprocessed = Dataset.from_dict(preprocess_medqa(all_data))

# Make sure the dataset is shuffled
dataset = all_data_preprocessed.shuffle(seed=42)

# Split into train (80%) and temp (20%)
train_testvalid = dataset.train_test_split(test_size=0.2, seed=42)

# Then split the 20% temp into valid and test (50/50 of 20% = 10% each)
test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

# Combine splits into a DatasetDict
dataset_dict = DatasetDict({
    'train': train_testvalid['train'],
    'validation': test_valid['train'],
    'test': test_valid['test']
})

train_dataset = dataset_dict["train"]
valid_dataset = dataset_dict["validation"]
test_dataset = dataset_dict["test"]

100%|██████████| 3404/3404 [00:27<00:00, 124.79it/s]


In [ ]:
# Verifying if we are working on the GPU
print(torch.cuda.is_available())  # Should return True
print(torch.cuda.device_count())  # Number of GPUs available
print(torch.cuda.get_device_name(0))  # GPU name
model.to("cuda")
print(next(model.parameters()).device)  # Should return: cuda:0

True
1
Tesla T4
cuda:0


In [ ]:
import evaluate

# Load metrics
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(p):
    predictions, labels = p
    preds = predictions.argmax(axis=1)
    accuracy = accuracy_metric.compute(predictions=preds, references=labels)
    return {"accuracy": accuracy["accuracy"]}

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    run_name="doctor_agent_2_1",
    output_dir="./",
    eval_strategy="epoch",     # Change from "epoch"
    save_strategy="epoch",
    metric_for_best_model="accuracy",
    greater_is_better=True,
    learning_rate=2e-5,
    fp16=True,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_strategy="steps",
    logging_steps=50,               # Logs loss every 50 steps
    load_best_model_at_end=True,
    report_to="none",
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics  # Pass the metric function
)

<ipython-input-16-60b783cb487e>:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.855000,0.600430,0.776471
2,0.685200,0.572005,0.785294
3,0.568900,0.497235,0.814706
4,0.412800,0.517995,0.832353
5,0.361300,0.480120,0.852941
6,0.451400,0.447801,0.850000
7,0.436100,0.456500,0.852941
8,0.398000,0.457639,0.850000
9,0.411100,0.452188,0.858824
10,0.377700,0.444116,0.850000


TrainOutput(global_step=6810, training_loss=0.5633780415712825, metrics={'train_runtime': 1568.5126, 'train_samples_per_second': 17.36, 'train_steps_per_second': 4.342, 'total_flos': 3.58222485508608e+16, 'train_loss': 0.5633780415712825, 'epoch': 10.0})

In [ ]:
print(trainer.state.best_model_checkpoint)

./medical_agent_simu/checkpoint-6129


Save model

In [ ]:
trainer.save_model("models/")
tokenizer.save_pretrained("models/")
print("Model saved")

Model saved


In [ ]:
!zip ./model.zip ./models/*

  adding: models/config.json (deflated 48%)
  adding: models/model.safetensors (deflated 7%)
  adding: models/special_tokens_map.json (deflated 80%)
  adding: models/tokenizer_config.json (deflated 74%)
  adding: models/training_args.bin (deflated 52%)
  adding: models/vocab.txt (deflated 54%)


In [ ]:
from google.colab import files
files.download('./model.zip')

Evaluation on test set

In [ ]:
results = trainer.evaluate(test_dataset)
name_file = "doctor_2.0_results"
with open(name_file + ".csv", "w") as f:
    for key, value in results.items():
        f.write(f"{key}: {value}\n")
print(results)

{'eval_loss': 0.3906288146972656, 'eval_accuracy': 0.8768328445747801, 'eval_runtime': 13.3066, 'eval_samples_per_second': 25.626, 'eval_steps_per_second': 6.463, 'epoch': 10.0}
